In [1]:
#!/usr/bin/env python3
"""
Download OPERA DIST-HLS products from CMR-STAC by bbox and date window.

Usage examples
--------------
# 1) DIST-ALERT between 2024-06-01 and 2024-06-30 over a bbox (minLon,minLat,maxLon,maxLat):
python download_dist_hls.py --product alert \
  --bbox -124.5 40.0 -123.7 40.7 \
  --start 2024-06-01 --end 2024-06-30 \
  --assets all --out ./dist_alert

# 2) DIST-ANN for calendar year 2024 (Jan 1–Dec 31):
python download_dist_hls.py --product ann \
  --bbox -70.5 -13.5 -69.5 -12.5 \
  --start 2024-01-01 --end 2024-12-31 \
  --assets core --out ./dist_ann

Notes
-----
- Requires an Earthdata Login (EL) and a ~/.netrc containing your EL creds:
  machine urs.earthdata.nasa.gov login <username> password <password>
- CMR-STAC provider: LPCLOUD.
- Assets are downloaded via HTTPS with URS auth.
"""

from __future__ import annotations
import argparse
import pathlib
import sys
import time
from typing import List, Dict

import requests
from requests.adapters import HTTPAdapter, Retry

CMR_STAC = "https://cmr.earthdata.nasa.gov/stac/LPCLOUD"
# STAC collection IDs (V1.1 as of 2025)
COLLECTIONS = {
    "alert": "OPERA_L3_DIST-ALERT-HLS_V1_1",
    "ann":   "OPERA_L3_DIST-ANN-HLS_V1_1",
}

# Which asset keys to fetch from each STAC Item
ASSET_GROUPS = {
    # "core": grab the most commonly used layers
    "core": [
        # ALERT:
        "vegetation_disturbance_alert",
        "vegetation_disturbance_confidence",
        # ANN:
        "vegetation_disturbance_status",
        "vegetation_disturbance_date_initial",
    ],
    # "all": fetch every listed asset in the item
    "all": None,
}

def make_session() -> requests.Session:
    s = requests.Session()
    retries = Retry(
        total=5, backoff_factor=0.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"])
    )
    s.mount("https://", HTTPAdapter(max_retries=retries))
    # URS auth via .netrc automatically, no explicit auth needed.
    s.trust_env = True
    return s

def stac_search(collection_id: str, bbox: List[float], start: str, end: str,
                limit: int = 200) -> List[Dict]:
    """
    Returns a list of STAC Items.
    """
    session = make_session()
    url = f"{CMR_STAC}/collections/{collection_id}/items"
    datetime_ = f"{start}T00:00:00Z/{end}T23:59:59Z"
    params = {
        "bbox": ",".join(map(str, bbox)),   # minx,miny,maxx,maxy
        "datetime": datetime_,
        "limit": min(limit, 200),
        "fields": "id,properties,assets,links,bbox"
    }

    items = []
    while True:
        r = session.get(url, params=params, timeout=60)
        r.raise_for_status()
        payload = r.json()
        feats = payload.get("features", [])
        items.extend(feats)

        # handle pagination via 'links' with rel=next
        next_link = None
        for ln in payload.get("links", []):
            if ln.get("rel") == "next":
                next_link = ln.get("href")
                break
        if not next_link:
            break
        url, params = next_link, None
        # be nice to the API
        time.sleep(0.2)
    return items

def choose_assets(item: Dict, group: str) -> Dict[str, Dict]:
    assets = item.get("assets", {})
    if ASSET_GROUPS[group] is None:
        return assets
    wanted = {}
    for key in ASSET_GROUPS[group]:
        if key in assets:
            wanted[key] = assets[key]
    # If nothing matched (e.g., using "core" on ALERT-only items), fall back to all assets
    if not wanted:
        return assets
    return wanted

def safe_name(name: str) -> str:
    return "".join(c if c.isalnum() or c in "._-+" else "_" for c in name)

def download_asset(session: requests.Session, href: str, out_path: pathlib.Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with session.get(href, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

def main():
    ap = argparse.ArgumentParser(description="Download OPERA DIST-HLS assets by bbox & dates")
    ap.add_argument("--product", choices=["alert", "ann"], required=True,
                    help="DIST product: alert (DIST-ALERT) or ann (DIST-ANN).")
    ap.add_argument("--bbox", nargs=4, type=float, required=True,
                    metavar=("MINLON","MINLAT","MAXLON","MAXLAT"),
                    help="Bounding box in WGS84 (lon/lat).")
    ap.add_argument("--start", required=True, help="Start date (YYYY-MM-DD)")
    ap.add_argument("--end", required=True, help="End date (YYYY-MM-DD)")
    ap.add_argument("--assets", choices=["core", "all"], default="core",
                    help="Which asset set to download from each item.")
    ap.add_argument("--out", default="./dist_hls_downloads",
                    help="Output directory.")
    ap.add_argument("--max", type=int, default=500,
                    help="Max number of items to retrieve.")
    args = ap.parse_args()

    col = COLLECTIONS[args.product]
    items = stac_search(col, args.bbox, args.start, args.end, limit=args.max)
    if not items:
        print("No items found for the given query.")
        return

    out_dir = pathlib.Path(args.out)
    session = make_session()

    print(f"Found {len(items)} items. Downloading assets ({args.assets}) to: {out_dir}")
    nfiles = 0
    for it in items:
        it_id = it.get("id", "unknown")
        sel_assets = choose_assets(it, args.assets)
        for akey, asset in sel_assets.items():
            href = asset.get("href")
            if not href or not href.startswith("http"):
                continue  # this script fetches HTTPS; S3 links can be handled separately if desired
            # Build filename: <itemid>__<assetkey>__<basename>
            base = pathlib.Path(href).name
            fname = f"{safe_name(it_id)}__{safe_name(akey)}__{base}"
            outp = out_dir / fname
            if outp.exists():
                continue
            try:
                download_asset(session, href, outp)
                nfiles += 1
            except Exception as e:
                print(f"[WARN] Failed: {href} -> {e}", file=sys.stderr)

    print(f"Done. Downloaded {nfiles} files.")

# if __name__ == "__main__":
#     main()


In [4]:
cases = {
  # "Mining": [
  #   {"lat": -5.773196381012654, "lon": -56.806566069338245, "date": "2025-06-29"},
  #   {"lat": -12.9890141517277,  "lon": -70.06357428871692, "date": "2025-04-18"},
  #   {"lat":  2.0185043672215914, "lon":  29.773492831188207, "date": "2024-12-02"},
  #   {"lat":  0.5632783027227907, "lon": 127.91000858625176, "date": "2024-10-06"},
  # ]
  #   "Logging": [
  #   {"lat": -7.614112272029289,  "lon":  -75.40825653555396, "date": "2024-09-06"},
  #   {"lat":  2.974193399509278,  "lon":   17.879592039849456, "date": "2025-02-28"},
  #   {"lat": 46.126040790060436,  "lon": -122.42122283481473, "date": "2024-11-30"},
  #   {"lat": 61.522819402112866,  "lon":   15.973959187506068, "date": "2025-05-15"},
  # ],
    "Road expansion": [
    {"lat": 35.42336004982615, "lon": 119.40456071405788, "date": "2025-04-30"},
    {"lat": 34.90684358345317, "lon": 115.46110157851172, "date": "2025-04-13"},
  ],
  "New construction": [
    {"lat": 38.89944199257163, "lon": -76.72312133749506, "date": "2024-12-31"},
  ]
  }

In [5]:
from datetime import datetime, timedelta
import pathlib
import sys

# --- your inputs ---
# cases = {
#   "Mining": [
#     {"lat": -5.773196381012654, "lon": -56.806566069338245, "date": "2025-06-29"},
#     {"lat": -12.9890141517277,  "lon": -70.06357428871692, "date": "2025-04-18"},
#     {"lat":  2.0185043672215914, "lon":  29.773492831188207, "date": "2024-12-02"},
#     {"lat":  0.5632783027227907, "lon": 127.91000858625176, "date": "2024-10-06"},
#   ]
# }

product_key = "alert"   # "alert" -> DIST-ALERT, or "ann" -> DIST-ANN
assets_group = "all"    # "core" or "all"
padding_deg = 0.1       # half-size of bbox in degrees
max_items = 10          # max items per query
base_out = pathlib.Path("DIST-HLS")

# --- helper ---
def plus_one_day(yyyy_mm_dd: str) -> str:
    d = datetime.strptime(yyyy_mm_dd, "%Y-%m-%d")
    return (d + timedelta(days=1)).strftime("%Y-%m-%d")

# --- main loop ---
col = COLLECTIONS[product_key]
session = make_session()

for group_name, pts in cases.items():
    for i, rec in enumerate(pts, start=1):
        lat, lon, start_date = rec["lat"], rec["lon"], rec["date"]
        end_date = plus_one_day(start_date)

        # bbox = [minLon, minLat, maxLon, maxLat]
        bbox = [
            lon - padding_deg,
            lat - padding_deg,
            lon + padding_deg,
            lat + padding_deg,
        ]

        print(f"\n[{group_name} #{i}] lat={lat:.6f}, lon={lon:.6f}, "
              f"date={start_date}..{end_date}, bbox={bbox}")

        try:
            items = stac_search(col, bbox, start_date, end_date, limit=max_items)
        except Exception as e:
            print(f"[ERROR] STAC search failed: {e}", file=sys.stderr)
            continue

        if not items:
            print("No items found for the given query.")
            continue

        out_dir = base_out / product_key / group_name / f"{start_date}_{i}"
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"Found {len(items)} items. Downloading assets ({assets_group}) to: {out_dir}")
        nfiles = 0

        for it in items:
            it_id = it.get("id", "unknown")
            sel_assets = choose_assets(it, assets_group)

            for akey, asset in sel_assets.items():
                href = asset.get("href")
                if not href or not href.startswith("http"):
                    # this script only handles HTTPS; skip s3:// here
                    continue

                base = pathlib.Path(href).name
                print("id  :", safe_name(it_id))
                print("key :", safe_name(akey))
                print("base:", base)

                # Example: only save GeoTIFFs (you can remove this if you want *everything*)
                if not base.endswith(".tif"):
                    continue

                outp = out_dir / base
                if outp.exists():
                    continue

                try:
                    download_asset(session, href, outp)
                    nfiles += 1
                except Exception as e:
                    print(f"[WARN] Failed: {href} -> {e}", file=sys.stderr)

        print(f"Downloaded {nfiles} files.")



[Road expansion #1] lat=35.423360, lon=119.404561, date=2025-04-30..2025-05-01, bbox=[119.30456071405789, 35.32336004982615, 119.50456071405787, 35.52336004982615]
Found 4 items. Downloading assets (all) to: DIST-HLS/alert/Road expansion/2025-04-30_1
id  : OPERA_L3_DIST-ALERT-HLS_T50SQE_20250430T023554Z_20250502T133306Z_L9_30_v1
key : browse
base: OPERA_L3_DIST-ALERT-HLS_T50SQE_20250430T023554Z_20250502T133306Z_L9_30_v1_VEG-DIST-STATUS.png
id  : OPERA_L3_DIST-ALERT-HLS_T50SQE_20250430T023554Z_20250502T133306Z_L9_30_v1
key : thumbnail_0
base: OPERA_L3_DIST-ALERT-HLS_T50SQE_20250430T023554Z_20250502T133306Z_L9_30_v1_VEG-DIST-STATUS.png
id  : OPERA_L3_DIST-ALERT-HLS_T50SQE_20250430T023554Z_20250502T133306Z_L9_30_v1
key : gov_lp-prod-protected_OPERA_L3_DIST-ALERT-HLS_V1_OPERA_L3_DIST-ALERT-HLS_T50SQE_20250430T023554Z_20250502T133306Z_L9_30_v1_OPERA_L3_DIST-ALERT-HLS_T50SQE_20250430T023554Z_20250502T133306Z_L9_30_v1_VEG-DIST-STATUS
base: OPERA_L3_DIST-ALERT-HLS_T50SQE_20250430T023554Z_2025